# Affinity-Based Agglomerative Clustering for Word Stroke Detection

**Approach**: Train a GradientBoosting classifier on stroke-pair features to predict P(same word), then use the predicted probabilities as an affinity matrix. Convert to distance (1 - probability) and run agglomerative clustering with average linkage.

**Why**: The previous pairwise classifier experiment used connected components — a single wrong edge merges entire words. Agglomerative clustering with average linkage is more robust: it considers the *average* affinity across all pairs in a candidate merge, not just the strongest single link.

**Evaluation**: Leave-one-page-out CV (6 folds), Jaccard-based cluster matching.

## 1. Data Loading

In [ ]:
import gzip
import json
import warnings
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from bs4 import BeautifulSoup as bs
from bs4 import XMLParsedAsHTMLWarning
from huggingface_hub import snapshot_download
from scipy.spatial.distance import cdist
from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

_TEXT_CLASSES = {"word", "digit", "mathematical_expression"}


@dataclass
class Stroke:
    x: np.ndarray
    y: np.ndarray


@dataclass
class PageData:
    page_id: str
    strokes: list[Stroke]
    word_groups: list[list[int]]


def _parse_strokes_per_layer(path: Path) -> list[list[list[Stroke]]]:
    with gzip.open(path, "r") as f:
        content = f.read().decode("utf-8")
    soup = bs(content, "lxml")
    pages = []
    for page in soup.find_all("page"):
        layers = []
        for layer in page.find_all("layer"):
            layer_strokes = []
            for stroke_el in layer.find_all("stroke"):
                coords = np.fromstring(stroke_el.text, sep=" ").reshape(-1, 2).T
                layer_strokes.append(Stroke(x=coords[0], y=coords[1]))
            layers.append(layer_strokes)
        pages.append(layers)
    return pages


def load_benchmark_pages() -> list[PageData]:
    local_dir = Path(
        snapshot_download("PellelNitram/xournalpp_htr_benchmark", repo_type="dataset")
    )
    data_dir = local_dir / "data"
    xopp_files = sorted(data_dir.glob("*.xopp")) + sorted(data_dir.glob("*.xoj"))

    pages = []
    for xopp_path in xopp_files:
        gt_path = xopp_path.with_suffix("").with_suffix(".gt.json")
        if not gt_path.exists():
            continue
        strokes_per_layer = _parse_strokes_per_layer(xopp_path)
        with open(gt_path) as f:
            gt = json.load(f)

        page_annotations: dict[int, list[list[int]]] = {}
        for ann in gt["annotations"]:
            if ann["class"] not in _TEXT_CLASSES:
                continue
            page_idx = ann["page_index"]
            layer_idx = ann["layer_index"]
            offset = sum(
                len(strokes_per_layer[page_idx][li]) for li in range(layer_idx)
            )
            global_indices = [offset + si for si in ann["stroke_indices"]]
            page_annotations.setdefault(page_idx, []).append(global_indices)

        for page_idx, layer_list in enumerate(strokes_per_layer):
            flat_strokes = [s for layer in layer_list for s in layer]
            if not flat_strokes:
                continue
            word_groups = page_annotations.get(page_idx, [])
            page_id = f"{xopp_path.stem}_p{page_idx}"
            pages.append(
                PageData(page_id=page_id, strokes=flat_strokes, word_groups=word_groups)
            )

    return pages

In [ ]:
pages = load_benchmark_pages()
for p in pages:
    n_word_strokes = len({si for wg in p.word_groups for si in wg})
    print(
        f"{p.page_id}: {len(p.strokes)} strokes, {len(p.word_groups)} words, {n_word_strokes} word strokes"
    )

## 2. Stroke & Pair Features

In [ ]:
@dataclass
class StrokeFeatures:
    bbox: tuple[float, float, float, float]
    centroid: tuple[float, float]
    n_points: int
    path_length: float
    bbox_width: float
    bbox_height: float
    bbox_area: float
    aspect_ratio: float


def compute_stroke_features(stroke: Stroke) -> StrokeFeatures:
    min_x, max_x = float(stroke.x.min()), float(stroke.x.max())
    min_y, max_y = float(stroke.y.min()), float(stroke.y.max())
    cx, cy = float(stroke.x.mean()), float(stroke.y.mean())
    n = len(stroke.x)
    diffs = np.sqrt(np.diff(stroke.x) ** 2 + np.diff(stroke.y) ** 2)
    path_length = float(diffs.sum())
    w = max_x - min_x
    h = max_y - min_y
    area = w * h
    aspect = w / h if h > 1e-6 else w / 1e-6
    return StrokeFeatures(
        bbox=(min_x, min_y, max_x, max_y),
        centroid=(cx, cy),
        n_points=n,
        path_length=path_length,
        bbox_width=w,
        bbox_height=h,
        bbox_area=area,
        aspect_ratio=aspect,
    )


FEATURE_NAMES = [
    "min_point_dist",
    "bbox_edge_dist",
    "centroid_dist",
    "horizontal_gap",
    "vertical_gap",
    "horizontal_overlap_ratio",
    "vertical_overlap_ratio",
    "centroid_angle",
    "stroke_index_diff",
    "area_ratio",
    "aspect_ratio_diff",
    "path_length_ratio",
    "n_points_ratio",
]


def compute_pair_features(
    stroke_a: Stroke,
    feat_a: StrokeFeatures,
    idx_a: int,
    stroke_b: Stroke,
    feat_b: StrokeFeatures,
    idx_b: int,
) -> np.ndarray:
    pts_a = np.column_stack([stroke_a.x, stroke_a.y])
    pts_b = np.column_stack([stroke_b.x, stroke_b.y])
    min_point_dist = cdist(pts_a, pts_b).min()

    ax0, ay0, ax1, ay1 = feat_a.bbox
    bx0, by0, bx1, by1 = feat_b.bbox
    dx = max(0.0, max(ax0 - bx1, bx0 - ax1))
    dy = max(0.0, max(ay0 - by1, by0 - ay1))
    bbox_edge_dist = np.sqrt(dx * dx + dy * dy)

    cx_diff = feat_a.centroid[0] - feat_b.centroid[0]
    cy_diff = feat_a.centroid[1] - feat_b.centroid[1]
    centroid_dist = np.sqrt(cx_diff**2 + cy_diff**2)

    horizontal_gap = dx
    vertical_gap = dy

    x_overlap = max(0.0, min(ax1, bx1) - max(ax0, bx0))
    min_width = min(feat_a.bbox_width, feat_b.bbox_width) + 1e-6
    horizontal_overlap_ratio = x_overlap / min_width

    y_overlap = max(0.0, min(ay1, by1) - max(ay0, by0))
    min_height = min(feat_a.bbox_height, feat_b.bbox_height) + 1e-6
    vertical_overlap_ratio = y_overlap / min_height

    centroid_angle = abs(np.arctan2(cy_diff, cx_diff + 1e-9))
    stroke_index_diff = abs(idx_a - idx_b)

    area_a = feat_a.bbox_area + 1e-6
    area_b = feat_b.bbox_area + 1e-6
    area_ratio = max(area_a, area_b) / min(area_a, area_b)
    aspect_ratio_diff = abs(feat_a.aspect_ratio - feat_b.aspect_ratio)

    pl_a = feat_a.path_length + 1e-6
    pl_b = feat_b.path_length + 1e-6
    path_length_ratio = max(pl_a, pl_b) / min(pl_a, pl_b)

    n_a = feat_a.n_points
    n_b = feat_b.n_points
    n_points_ratio = max(n_a, n_b) / max(min(n_a, n_b), 1)

    return np.array(
        [
            min_point_dist,
            bbox_edge_dist,
            centroid_dist,
            horizontal_gap,
            vertical_gap,
            horizontal_overlap_ratio,
            vertical_overlap_ratio,
            centroid_angle,
            stroke_index_diff,
            area_ratio,
            aspect_ratio_diff,
            path_length_ratio,
            n_points_ratio,
        ]
    )

## 3. Build Training Data

In [ ]:
def build_page_data(
    page: PageData,
) -> tuple[np.ndarray, np.ndarray, list[tuple[int, int]]]:
    stroke_feats = [compute_stroke_features(s) for s in page.strokes]

    stroke_to_word = {}
    for wi, wg in enumerate(page.word_groups):
        for si in wg:
            stroke_to_word[si] = wi

    n = len(page.strokes)
    pairs = list(combinations(range(n), 2))

    X = np.zeros((len(pairs), len(FEATURE_NAMES)))
    y = np.zeros(len(pairs), dtype=int)

    for k, (i, j) in enumerate(pairs):
        X[k] = compute_pair_features(
            page.strokes[i],
            stroke_feats[i],
            i,
            page.strokes[j],
            stroke_feats[j],
            j,
        )
        wi = stroke_to_word.get(i, -1)
        wj = stroke_to_word.get(j, -1)
        y[k] = 1 if (wi == wj and wi != -1) else 0

    return X, y, pairs


print("Building pairwise data per page...")
page_data = {}
for page in pages:
    X, y, pairs = build_page_data(page)
    page_data[page.page_id] = (X, y, pairs)
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    print(
        f"  {page.page_id}: {len(pairs)} pairs, {n_pos} same-word, {n_neg} diff-word, ratio {n_neg/(n_pos+1e-9):.1f}:1"
    )

## 4. Evaluation Metrics

In [ ]:
@dataclass
class GroupingMetrics:
    precision: float
    recall: float
    f1: float
    stroke_accuracy: float
    n_gt_words: int
    n_pred_clusters: int


def evaluate_grouping(
    predicted_clusters: list[set[int]],
    gt_word_groups: list[list[int]],
    n_strokes: int,
    jaccard_threshold: float = 0.5,
) -> GroupingMetrics:
    gt_sets = [set(wg) for wg in gt_word_groups]

    candidates = []
    for i, pred in enumerate(predicted_clusters):
        for j, gt in enumerate(gt_sets):
            intersection = len(pred & gt)
            union = len(pred | gt)
            if union > 0:
                jaccard = intersection / union
                if jaccard >= jaccard_threshold:
                    candidates.append((jaccard, i, j))

    candidates.sort(key=lambda x: x[0], reverse=True)

    matched_pred = set()
    matched_gt = set()
    for _, pi, gi in candidates:
        if pi not in matched_pred and gi not in matched_gt:
            matched_pred.add(pi)
            matched_gt.add(gi)

    n_matched = len(matched_pred)
    n_pred = len(predicted_clusters)
    n_gt = len(gt_sets)

    precision = n_matched / n_pred if n_pred > 0 else 0.0
    recall = n_matched / n_gt if n_gt > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    stroke_to_gt = {}
    for gi, gt in enumerate(gt_sets):
        for si in gt:
            stroke_to_gt[si] = gi

    stroke_to_pred = {}
    for pi, pred in enumerate(predicted_clusters):
        for si in pred:
            stroke_to_pred[si] = pi

    correct = 0
    total_word_strokes = len(stroke_to_gt)
    for si, gi in stroke_to_gt.items():
        if si in stroke_to_pred:
            pi = stroke_to_pred[si]
            pred_set = predicted_clusters[pi]
            gt_set = gt_sets[gi]
            intersection = len(pred_set & gt_set)
            union = len(pred_set | gt_set)
            if union > 0 and intersection / union >= jaccard_threshold:
                correct += 1

    stroke_accuracy = correct / total_word_strokes if total_word_strokes > 0 else 0.0

    return GroupingMetrics(
        precision=precision,
        recall=recall,
        f1=f1,
        stroke_accuracy=stroke_accuracy,
        n_gt_words=n_gt,
        n_pred_clusters=n_pred,
    )

## 5. Affinity-Based Clustering

Build an NxN distance matrix from classifier probabilities: distance = 1 - P(same word).
Then run agglomerative clustering with average linkage.

In [ ]:
def build_distance_matrix(
    pairs: list[tuple[int, int]],
    proba: np.ndarray,
    n_strokes: int,
) -> np.ndarray:
    """Build an NxN symmetric distance matrix from pairwise probabilities."""
    dist = np.ones((n_strokes, n_strokes))
    np.fill_diagonal(dist, 0.0)
    for (i, j), p in zip(pairs, proba, strict=False):
        dist[i, j] = 1.0 - p
        dist[j, i] = 1.0 - p
    return dist


def affinity_cluster(
    pairs: list[tuple[int, int]],
    proba: np.ndarray,
    n_strokes: int,
    distance_threshold: float,
) -> list[set[int]]:
    """Cluster strokes using agglomerative clustering on probability-based distance."""
    if n_strokes == 0:
        return []
    if n_strokes == 1:
        return [{0}]

    dist = build_distance_matrix(pairs, proba, n_strokes)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=distance_threshold,
    )
    labels = clustering.fit_predict(dist)

    clusters: dict[int, set[int]] = {}
    for i, label in enumerate(labels):
        clusters.setdefault(label, set()).add(i)

    return list(clusters.values())

## 6. LOPO Cross-Validation

Train GradientBoosting on 5 pages, predict probabilities, build affinity distance matrix, tune distance threshold on training pages, evaluate on held-out page.

In [ ]:
def tune_distance_threshold(
    clf,
    scaler,
    train_pages: list[PageData],
    page_data: dict,
    candidates: np.ndarray | None = None,
) -> float:
    """Find distance threshold that maximises mean F1 on training pages."""
    if candidates is None:
        candidates = np.arange(0.1, 0.95, 0.05)

    best_threshold = 0.5
    best_f1 = -1.0

    for threshold in candidates:
        f1s = []
        for page in train_pages:
            X_val, _, pairs_val = page_data[page.page_id]
            X_val_scaled = scaler.transform(X_val)
            proba = clf.predict_proba(X_val_scaled)[:, 1]
            clusters = affinity_cluster(pairs_val, proba, len(page.strokes), threshold)
            metrics = evaluate_grouping(clusters, page.word_groups, len(page.strokes))
            f1s.append(metrics.f1)
        mean_f1 = np.mean(f1s)
        if mean_f1 > best_f1:
            best_f1 = mean_f1
            best_threshold = threshold

    return float(best_threshold)


results = []
for i, test_page in enumerate(pages):
    train_pages = pages[:i] + pages[i + 1 :]

    X_train = np.vstack([page_data[p.page_id][0] for p in train_pages])
    y_train = np.concatenate([page_data[p.page_id][1] for p in train_pages])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    clf = GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42
    )
    clf.fit(X_train_scaled, y_train)

    threshold = tune_distance_threshold(clf, scaler, train_pages, page_data)

    X_test, _, pairs_test = page_data[test_page.page_id]
    X_test_scaled = scaler.transform(X_test)
    proba = clf.predict_proba(X_test_scaled)[:, 1]

    clusters = affinity_cluster(pairs_test, proba, len(test_page.strokes), threshold)
    metrics = evaluate_grouping(clusters, test_page.word_groups, len(test_page.strokes))

    results.append((test_page.page_id, threshold, metrics, clusters))
    print(
        f"Fold {i+1}/6: {test_page.page_id} — threshold={threshold:.2f}, F1={metrics.f1:.3f}"
    )

## 7. Results

In [ ]:
def print_results_table(results, title):
    print(f"\n{'=' * 70}")
    print(title)
    print("=" * 70)

    headers = [
        "Page",
        "Threshold",
        "Precision",
        "Recall",
        "F1",
        "Stroke Acc",
        "GT Words",
        "Pred Clusters",
    ]
    rows = []
    for page_id, threshold, m, _ in results:
        rows.append(
            [
                page_id,
                f"{threshold:.2f}",
                f"{m.precision:.3f}",
                f"{m.recall:.3f}",
                f"{m.f1:.3f}",
                f"{m.stroke_accuracy:.3f}",
                m.n_gt_words,
                m.n_pred_clusters,
            ]
        )

    precisions = [m.precision for _, _, m, _ in results]
    recalls = [m.recall for _, _, m, _ in results]
    f1s = [m.f1 for _, _, m, _ in results]
    stroke_accs = [m.stroke_accuracy for _, _, m, _ in results]

    rows.append(
        [
            "MEAN",
            "",
            f"{np.mean(precisions):.3f} ± {np.std(precisions):.3f}",
            f"{np.mean(recalls):.3f} ± {np.std(recalls):.3f}",
            f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
            f"{np.mean(stroke_accs):.3f} ± {np.std(stroke_accs):.3f}",
            "",
            "",
        ]
    )

    print(tabulate(rows, headers=headers, tablefmt="grid"))


print_results_table(
    results, "Affinity-Based Agglomerative Clustering (GB + average linkage)"
)


# Comparison summary
def summarize(name, results):
    f1s = [m.f1 for _, _, m, _ in results]
    precisions = [m.precision for _, _, m, _ in results]
    recalls = [m.recall for _, _, m, _ in results]
    stroke_accs = [m.stroke_accuracy for _, _, m, _ in results]
    return [
        name,
        f"{np.mean(precisions):.3f} ± {np.std(precisions):.3f}",
        f"{np.mean(recalls):.3f} ± {np.std(recalls):.3f}",
        f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
        f"{np.mean(stroke_accs):.3f} ± {np.std(stroke_accs):.3f}",
    ]


print("\n" + "=" * 70)
print("COMPARISON SUMMARY")
print("=" * 70)

comparison = [
    [
        "Spatial Baseline",
        "0.762 ± 0.175",
        "0.734 ± 0.213",
        "0.733 ± 0.178",
        "0.641 ± 0.259",
    ],
    [
        "GB + Conn. Components",
        "0.738 ± 0.219",
        "0.741 ± 0.318",
        "0.719 ± 0.288",
        "0.768 ± 0.286",
    ],
    summarize("GB + Affinity Clustering", results),
]

print(
    tabulate(
        comparison,
        headers=["Method", "Precision", "Recall", "F1", "Stroke Acc"],
        tablefmt="grid",
    )
)

## 8. Feature Importance

In [ ]:
X_all = np.vstack([page_data[p.page_id][0] for p in pages])
y_all = np.concatenate([page_data[p.page_id][1] for p in pages])

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

clf_inspect = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42
)
clf_inspect.fit(X_all_scaled, y_all)

importances = clf_inspect.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(FEATURE_NAMES)), importances[sorted_idx])
ax.set_yticks(range(len(FEATURE_NAMES)))
ax.set_yticklabels([FEATURE_NAMES[i] for i in sorted_idx])
ax.set_xlabel("Feature Importance")
ax.set_title("Gradient Boosting Feature Importances")
plt.tight_layout()
plt.show()

## 9. Visualize Predictions vs GT

In [ ]:
def plot_comparison(page: PageData, clusters: list[set[int]], title: str = ""):
    fig, (ax_gt, ax_pred) = plt.subplots(1, 2, figsize=(16, 8))
    cmap = plt.colormaps["tab20"]

    ax_gt.set_title("Ground Truth")
    word_stroke_set = {si for wg in page.word_groups for si in wg}
    for gi, wg in enumerate(page.word_groups):
        color = cmap(gi % 20)
        for si in wg:
            s = page.strokes[si]
            ax_gt.plot(s.x, -s.y, color=color, linewidth=1.5)
    for si, s in enumerate(page.strokes):
        if si not in word_stroke_set:
            ax_gt.plot(s.x, -s.y, color="lightgray", linewidth=0.5)
    ax_gt.set_aspect("equal")
    ax_gt.axis("off")

    ax_pred.set_title("Predicted Clusters")
    clustered_strokes = {si for c in clusters for si in c}
    for ci, cluster in enumerate(clusters):
        color = cmap(ci % 20)
        for si in cluster:
            s = page.strokes[si]
            ax_pred.plot(s.x, -s.y, color=color, linewidth=1.5)
    for si, s in enumerate(page.strokes):
        if si not in clustered_strokes:
            ax_pred.plot(s.x, -s.y, color="lightgray", linewidth=0.5)
    ax_pred.set_aspect("equal")
    ax_pred.axis("off")

    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


for page_id, threshold, m, clusters in results:
    page = next(p for p in pages if p.page_id == page_id)
    plot_comparison(
        page, clusters, f"{page_id} (threshold={threshold:.2f}, F1={m.f1:.3f})"
    )